# 16. Ingestão da base real
Desenvolvimento da montar_base, o passo que baixa Ibovespa e CDI e enche o banco. Requisitos F1 e NF6.

In [ ]:
import sys, os, tempfile
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import os
from app import dal

import pandas as _pd
print(f'kernel: {sys.executable}  (pandas {_pd.__version__})')
if 'venv' not in sys.executable.replace(os.sep, '/').split('/'):
    print('  ATENCAO: este kernel NAO e o venv do projeto; os resultados podem diferir.')


kernel: C:\Users\DELL\Desktop\TCC\optimal-trading-strategies-stochastic-discrete-time-environmente-brazil\venv\Scripts\python.exe  (pandas 2.2.3)


## Desenvolvimento

A função abaixo foi desenvolvida aqui e depois passou para app/ingestao.py.

In [2]:
BANCO_PADRAO = {"1mo": "data/mercado.db", "1d": "data/mercado_diario.db"}

In [ ]:
def montar_base(db_path: str | None = None, inicio: str = "2000-01-01",
                fim: str | None = None, frequencia: str = "1mo") -> dict:
    """
    Baixa o Ibovespa no Yahoo e o CDI no Banco Central e grava no SQLite.

    O db_path e o caminho do banco; se vier None, usa o padrao da frequencia.
    A frequencia e "1mo" pra mensal ou "1d" pra diario.

    Devolve um dicionario com db_path, frequencia, n_periodos e periodo, que
    traz a primeira e a ultima data.
    """
    if frequencia not in BANCO_PADRAO:
        raise ValueError(
            f"frequência desconhecida: {frequencia!r} (use {sorted(BANCO_PADRAO)})."
        )
    if fim is not None and fim < inicio:
        raise ValueError(f"o início ({inicio}) vem depois do fim ({fim}).")
    if db_path is None:
        db_path = BANCO_PADRAO[frequencia]
    os.makedirs(os.path.dirname(db_path) or ".", exist_ok=True)

    # 1. o Ibovespa, que sao os niveis do indice (tabela 'ibovespa')
    precos = dal.baixar_precos(["^BVSP"], inicio, fim, frequencia=frequencia)
    precos = precos.rename(columns={precos.columns[1]: "fechamento"})
    dal.gravar_sqlite(precos, db_path, "ibovespa")

    # 2. o CDI, que ja e uma taxa por periodo (tabela 'cdi')
    cdi = dal.baixar_cdi_bcb(inicio, fim, frequencia=frequencia)
    dal.gravar_sqlite(cdi, db_path, "cdi")

    # 3. os retornos alinhados por data (tabela 'retornos').
    ret_ibov = dal.calcular_retornos(precos.rename(columns={"fechamento": "ibov"}))
    retornos = ret_ibov.merge(cdi, on="data", how="inner")
    if retornos.empty:
        raise ValueError("Nao tem nenhuma data em comum entre Ibovespa e CDI. Confira o periodo.")
    dal.gravar_sqlite(retornos, db_path, "retornos")

    return {"db_path": db_path, "frequencia": frequencia, "n_periodos": len(retornos),
            "periodo": (retornos["data"].iloc[0], retornos["data"].iloc[-1])}

**Teste**: monta o banco (download real, protegido).

In [4]:
import tempfile

In [5]:
try:
    dbm = os.path.join(tempfile.gettempdir(), 'dev16.db')
    info = montar_base(dbm, '2020-01-01', '2020-06-01')

    print('montar_base ->', info)
    print(dal.ler_sqlite(dbm,'retornos'))

    assert set(dal.ler_sqlite(dbm,'retornos').columns) == {'data','ibov','cdi'}
    os.remove(dbm)
except Exception as e:
    print('offline:', type(e).__name__, e)

montar_base -> {'db_path': 'C:\\Users\\DELL\\AppData\\Local\\Temp\\dev16.db', 'frequencia': '1mo', 'n_periodos': 4, 'periodo': ('2020-02', '2020-05')}
      data      ibov     cdi
0  2020-02 -0.084291  0.0029
1  2020-03 -0.299044  0.0034
2  2020-04  0.102520  0.0028
3  2020-05  0.085671  0.0024


In [6]:
# --- ingestao DIARIA ---
from app import ingestao as _ing
assert _ing.BANCO_PADRAO == {'1mo': 'data/mercado.db', '1d': 'data/mercado_diario.db'}

In [ ]:
# (a) banco diario versionado no repo: valida a estrutura sem tocar na rede
dbd = os.path.join(RAIZ, 'data', 'mercado_diario.db')

if os.path.exists(dbd):
    rd = dal.ler_sqlite(dbd, 'retornos')

    print(f'base diaria: {len(rd)} pregoes, {rd["data"].iloc[0]} a {rd["data"].iloc[-1]}')

    assert set(rd.columns) == {'data', 'ibov', 'cdi'}
    assert rd['data'].str.match(r'^\d{4}-\d{2}-\d{2}$').all()
    assert rd['data'].is_unique and rd['data'].is_monotonic_increasing
    assert not rd.isna().any().any() and len(rd) > 500
else:
    print('data/mercado_diario.db ausente -> python -m app.ingestao 2022-05-22 --diario')

base diaria: 1050 pregoes, 2022-05-24 a 2026-08-05


In [ ]:
# (b) download diario de verdade, se houver rede
try:
    dbd_tmp = os.path.join(tempfile.gettempdir(), 'dev16d.db')
    info_d = _ing.montar_base(dbd_tmp, '2024-01-01', '2024-03-01', frequencia='1d')

    print('montar_base(1d) ->', info_d)

    assert info_d['frequencia'] == '1d' and info_d['n_periodos'] > 20
    assert len(info_d['periodo'][0]) == 10
    os.remove(dbd_tmp)
except Exception as e:
    print('offline:', type(e).__name__, e)

montar_base(1d) -> {'db_path': 'C:\\Users\\DELL\\AppData\\Local\\Temp\\dev16d.db', 'frequencia': '1d', 'n_periodos': 40, 'periodo': ('2024-01-03', '2024-02-29')}


**Teste** — as validações de entrada, que acontecem antes de qualquer download e por isso rodam offline.

In [9]:
# a checagem vem antes do makedirs e da rede: nada e criado nem baixado
alvo = os.path.join(tempfile.gettempdir(), 'nao_deve_existir16.db')
for kwargs, esperado in [({'frequencia': 'semanal'}, 'frequência desconhecida'),
                         ({'inicio': '2020-06-01', 'fim': '2020-01-01'}, 'vem depois do fim')]:
    try:
        _ing.montar_base(alvo, **kwargs)
        raise AssertionError('passou sem erro: %s' % kwargs)
    except ValueError as e:
        assert esperado in str(e), e
        print('recusado:', e)
assert not os.path.exists(alvo)

recusado: frequência desconhecida: 'semanal' (use ['1d', '1mo']).
recusado: o início (2020-06-01) vem depois do fim (2020-01-01).
